In [21]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:

import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/ecommerce")

customer = pd.read_csv(r"C:\Users\HP\Desktop\Ecommerce Analysis\customer_master.csv")
order_items = pd.read_csv(r"C:\Users\HP\Desktop\Ecommerce Analysis\order_items.csv")
products = pd.read_csv(r"C:\Users\HP\Desktop\Ecommerce Analysis\product_catalog.csv")
sales = pd.read_csv(r"C:\Users\HP\Desktop\Ecommerce Analysis\ecommerce_sales_customer_analytics_150k.csv")

customer.to_sql("customer_master", engine, if_exists="replace", index=False)
order_items.to_sql("order_items", engine, if_exists="replace", index=False)
products.to_sql("product_catalog", engine, if_exists="replace", index=False)
sales.to_sql("sales", engine, if_exists="replace", index=False)

print("All tables loaded successfully!")

All tables loaded successfully!


In [5]:
# Check the number rows in Tables
query1 = """
SELECT 'customer_master' AS table_name, COUNT(*) AS Total_Rows FROM customer_master
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'product_catalog', COUNT(*) FROM product_catalog
UNION ALL
SELECT 'sales', COUNT(*) FROM sales;
"""
df1 = pd.read_sql(query1, engine)
print(df1)

query2 = """
SELECT *FROM sales
LIMIT 10;
"""
Read = pd.read_sql(query2, engine)
print(Read)

# Check columns of sales table
query3 = """
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'sales';
"""
Verfy = pd.read_sql(query3, engine)
print(Verfy)

        table_name  total_rows
0  product_catalog        1175
1  customer_master       25000
2            sales      138116
3      order_items      397569
     order_id  order_date order_time order_status sales_channel  customer_id  \
0  ORD-301242  2023-11-06   16:37:47    Completed    Mobile App  CUST-003102   
1  ORD-773460  2025-12-24   01:22:36    Completed       Website  CUST-003124   
2  ORD-449374  2021-07-05   14:24:21    Completed    Mobile App  CUST-012496   
3  ORD-567636  2023-01-21   07:20:26    Completed  Social Media  CUST-023928   
4  ORD-820028  2022-05-13   09:46:21    Completed  Social Media  CUST-012730   
5  ORD-208728  2021-02-10   10:05:47      Pending       Website  CUST-021468   
6  ORD-686230  2021-05-03   02:55:44    Completed    Mobile App  CUST-023734   
7  ORD-480470  2022-10-13   21:19:50    Completed    Mobile App  CUST-020081   
8  ORD-417329  2022-12-06   17:22:52    Completed    Mobile App  CUST-011921   
9  ORD-961581  2023-09-19   18:38:32    Compl

In [6]:
# OVerall Business Matrix
query4 = """
SELECT
     COUNT(*) AS Total_Orders,
     ROUND(SUM(net_sales)::numeric, 2) As TotalRevenue,
     ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
     ROUND(AVG(net_sales)::numeric, 2) As Average_Order_value,
    ROUND(AVG(profit_margin_percentage)::numeric, 2) AS avg_profit_margin
FROM sales
"""
Overall = pd.read_sql(query4, engine)
print(Overall)




   total_orders  totalrevenue  total_profit  average_order_value  \
0        138116  1.771343e+08   76146395.76               1282.5   

   avg_profit_margin  
0              45.07  


In [ ]:
s

# Revenue and Profit By Sales Channel
query7 = """
SELECT sales_channel,
		ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
    COUNT(*) AS Total_Orders
FROM sales
GROUP BY Sales_channel
ORDER BY Sales_channel DESC;
"""
Sales_channel = pd.read_sql(query7, engine)
print(Sales_channel)

# Revenue and profit by Marketing Channel
query8 = """
SELECT marketing_channel,
	ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
    COUNT(*) AS Total_Orders
FROM sales
GROUP BY marketing_channel
ORDER BY marketing_channel DESC;
"""
Markeing_channel = pd.read_sql(query8, engine)
print(Markeing_channel)

    region  total_revenue  total_profit  total_orders
0    South    55587994.98   23343946.17         43990
1  Central    37171107.17   15572868.84         29627
2     West    34755096.47   15647252.47         26273
3     East    27054887.88   11179366.94         21602
4    North    22565177.24   10402961.34         16624
  customer_segment  total_revenue  total_profit  total_orders
0         Consumer    97201007.90   42819790.02         75636
1          Premium    44269794.13   18177727.78         34746
2              VIP    17903164.81    7366352.68         13957
3         Business    17760296.90    7782525.28         13777
  sales_channel  total_revenue  total_profit  total_orders
0       Website    61854736.08   26639693.01         48237
1  Social Media    17748554.91    7641363.37         13863
2    Mobile App    70834782.53   30451112.22         55318
3   Marketplace    26696190.22   11414227.16         20698
  marketing_channel  total_revenue  total_profit  total_orders
0       

In [8]:
# Monthly Performance
query9 = """
SELECT TO_CHAR(order_date::date,'YYYY-MM') AS month,
	ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit
FROM sales
GROUP by month
ORDER BY month;
"""
Monthly_Perfomance = pd.read_sql(query9, engine)
print(Monthly_Perfomance)

#Product Performance
query10 = """
SELECT p.product_id, p.product_name, 
	ROUND(SUM(o.net_sales)::numeric, 2) AS Total_Prodduct_sales,
	ROUND(SUM(o.profit)::numeric, 2) AS Total_Product_profit
FROM order_items o
LEFT JOIN product_catalog p
	ON p.product_id = o.product_id
GROUP BY p.product_id, p.product_name
ORDER BY Total_Product_profit DESC
LIMIT 15;
"""
Product_Performace = pd.read_sql(query10, engine)
print(Product_Performace)


      month  total_revenue  total_profit
0   2021-01     2374415.22    1129764.43
1   2021-02     1905413.15     900452.55
2   2021-03     2753256.18    1298951.77
3   2021-04     2469384.78    1178381.90
4   2021-05     2546555.37    1219361.44
5   2021-06     2710208.64    1292627.89
6   2021-07     2894416.72    1143965.61
7   2021-08     3220659.48    1526095.90
8   2021-09     2720931.59    1224683.15
9   2021-10     2991778.76    1413407.38
10  2021-11     4234660.71    1371164.82
11  2021-12     4900970.07    1626795.03
12  2022-01     2442770.60    1168953.10
13  2022-02     2029412.22     968538.06
14  2022-03     2682800.73    1269999.83
15  2022-04     2516740.73    1197099.96
16  2022-05     2529145.45    1211110.70
17  2022-06     2697622.96    1274750.83
18  2022-07     2765292.19    1106041.95
19  2022-08     2947796.56    1408179.47
20  2022-09     2750776.22    1238645.19
21  2022-10     2925151.89    1393120.46
22  2022-11     4405731.78    1424196.88
23  2022-12     

In [9]:
# Repeat customers vs New customers
query11 = """
SELECT is_repeat_customer, ROUND(SUM(profit)::numeric, 2) AS Total_Product_profit,
		ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
	ROUND(AVG(net_sales)::numeric, 2) AS AVerage_Order_value,
    COUNT(*) AS Total_Orders	
FROM sales
GROUP BY is_repeat_customer;
"""
Repeat = pd.read_sql(query11,engine)
print(Repeat)

   is_repeat_customer  total_product_profit  total_revenue  total_profit  \
0               False             192740.73   4.354568e+05     192740.73   
1                True           75953655.03   1.766988e+08   75953655.03   

   average_order_value  total_orders  
0              1303.76           334  
1              1282.45        137782  


In [10]:
# Return and Delivery Performance
#Returns
query12 = """
SELECT return_status, COUNT(*) AS Total_Orders,
		ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
		ROUND(COUNT(*) * 100 / sum(COUNT(*))OVER (),2) AS percentage_of_orders
FROM sales
GROUP BY return_status
ORDER BY Total_orders DESC;
"""
Returns = pd.read_sql(query12, engine)
print(Returns)

# Delivery Status
query13 = """
SELECT delivery_status, COUNT(*) AS Total_Orders,
		ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
		ROUND(COUNT(*) * 100 / sum(COUNT(*))OVER (),2) AS percentage_of_orders
FROM sales
GROUP BY delivery_status
ORDER BY Total_orders DESC;
"""
Status = pd.read_sql(query13, engine)
print(Status)

#Reasons
query14 = """
SELECT return_reason, COUNT(*) AS Total_Returns
FROM sales
WHERE return_status =  'Returned'
GROUP BY return_reason
ORDER BY Total_Returns DESC;
"""
Reasoning = pd.read_sql(query14, engine)
print(Reasoning)

  return_status  total_orders  total_revenue  percentage_of_orders
0           NaN        128654   1.639616e+08                 93.15
1      Returned          9462   1.317267e+07                  6.85
  delivery_status  total_orders  total_revenue  percentage_of_orders
0         On Time         86916   1.102959e+08                 62.93
1       Cancelled         24557   3.293843e+07                 17.78
2         Delayed         16886   2.145506e+07                 12.23
3           Early          9757   1.244488e+07                  7.06
             return_reason  total_returns
0            Wrong Product           1237
1                    Other           1208
2             Changed Mind           1204
3               Size Issue           1182
4        Defective Product           1174
5            Late Delivery           1172
6  Product Not as Expected           1147
7          Damaged Product           1138


In [15]:
## Discount Impact

# Overall Discount Summary
query15 = """
SELECT ROUND(AVG(discount_amount)::numeric ,2) AS Average_Discount_Amount,
ROUND(SUM(discount_amount)::numeric, 2) AS total_discount_given,
ROUND(AVG(profit_margin_percentage)::numeric, 2) AS Average_Profit_Margin
FROM sales;
"""
Discount_Summary = pd.read_sql(query15, engine)
print(Discount_Summary)

# Discount v/s No Discount
query16 = """
SELECT 
    CASE 
        WHEN discount_amount > 0 THEN 'With Discount'
        ELSE 'No Discount'
    END AS discount_status,
    COUNT(*) AS total_orders,
    ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
    ROUND(SUM(profit)::numeric, 2) AS total_profit,
    ROUND(AVG(profit_margin_percentage)::numeric, 2) AS avg_profit_margin
FROM sales
GROUP BY discount_status
ORDER BY total_revenue DESC;
"""
Discount_Diff = pd.read_sql(query16, engine)
print(Discount_Diff)

   average_discount_amount  total_discount_given  average_profit_margin
0                   237.62           32819346.82                  45.07
  discount_status  total_orders  total_revenue  total_profit  \
0   With Discount        120252   1.526047e+08   63246422.24   
1     No Discount         17864   2.452954e+07   12899973.52   

   avg_profit_margin  
0              43.72  
1              54.11  


In [23]:
# Discount Range Differnce
query17 = text("""
SELECT 
    CASE 
        WHEN discount_amount = 0 THEN '0%'
        WHEN discount_amount > 0 AND discount_amount <= 100 THEN 'Low (0-100)'
        WHEN discount_amount > 100 AND discount_amount <= 300 THEN 'Medium (100-300)'
        ELSE 'High (300+)'
    END AS discount_range,
    COUNT(*) AS total_orders,
    ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
    ROUND(SUM(profit)::numeric, 2) AS total_profit,
    ROUND(AVG(profit_margin_percentage)::numeric, 2) AS avg_profit_margin
FROM sales
GROUP BY discount_range
ORDER BY total_revenue DESC;
""")
Discount_Range = pd.read_sql_query(query17, engine)
print(Discount_Range)

     discount_range  total_orders  total_revenue  total_profit  \
0       High (300+)         32075    74130029.01   25623148.42   
1  Medium (100-300)         38103    50992440.04   23709915.49   
2       Low (0-100)         50074    27482249.90   13913358.33   
3                0%         17864    24529544.79   12899973.52   

   avg_profit_margin  
0              32.73  
1              44.88  
2              49.88  
3              54.11  
